# Quality control and removal of host reads

Alba Nagel-González and Jacobo de la Cuesta-Zuluaga. August 2026.

The aim of this notebook is to perform sequence quality control, filtering, and 
removal of host reads from raw metagenome sequences. For this, we will use the 
`nf-core` pipeline [`detaxizer`](https://nf-co.re/detaxizer/1.3.0).

Once QC'd, the clean sequences can be used for diverse downstream analyses.


## A note on host removal and filtering

Whenever we use metagenome sequences, the first step should always be to remove low quality reads and
reads that (likely) come from the host; usually, human or mouse. Once you have clean reads, these can
be used for any sort of analysis, such as metagenome assembly or obtaining taxonomic and functional
profiles. Many pipelines come with an integrated QC step, but this is not always the case, that's why
we're running this pipeline independently.

**Note** that for submission of sequences to the ENA/SRA, these have to be cleaned of host reads but
not trimmed. This means that *you should not* submit the raw sequences nor the output of this
pipeline. You need to process the raw sequences again before submission. Fortunately, the recently
released pipeline [seqsubmit](https://nf-co.re/seqsubmit/) from `nf-core` can help with the whole
process, including the removal of human reads. Alternatively, we can run the `detaxizer` pipeline
again on the raw reads skipping the cleaning step. For this, remove the `--preprocessing` argument
from the pipeline command (see below).

## Before we start

This notebook requires `conda` and the `Nextflow` and `VScode` environments of this repo.
Instructions to install `conda` are [here](https://conda.io/projects/conda/en/latest/user-guide/install/index.html); 
if you are on the M3 cluster you should already have it.

You can install the required environments once with:

```bash
cd Path/To/Metemgee
conda env create -f envs/Nextflow.yaml
conda env create -f envs/VScode.yaml
```
The notebooks are written in **R**, not Python. In VSCode, click the kernel selector on the top right
and pick the R kernel from the `VScode` environment.

## What you'll need to change

These are the only values you have to edit. Everything else can stay as it is.

| Variable | Where | What to put there |
|---|---|---|
| `base_dir` | Load libraries and set paths | Directory where your input and output will live |
| `repo_dir` | Load libraries and set paths | Where you cloned this repository, for the config file |
| `seq_dir` | Load libraries and set paths | Only if your reads are not inside `base_dir` |
| `str_remove` patterns | Create Samples file | Only if your file names differ from the example |
| `--tax2filter`, `--genome` | Execute pipeline | Only if your host is not human |

## Load libraries and set paths

First, we'll set up the libraries and the work directory where we'll save our files.

In [ ]:
# Libraries
library(tidyverse)
library(conflicted)

In [ ]:
# Housekeeping: tells R which `filter` function to use when more than one
# package provides one. Nothing to change here.
conflicts_prefer(dplyr::filter)

The following chunk will define the directories where the data is stored and where the output will be
saved. The present example assumes everything will be contained in the same directory: `base_dir`.
This might be different in your particular case, for example, if your sequences are stored on a
centralized directory or you have multiple runs stored in different folders. You can change this
accordingly.

`base_dir` has to exist already; the chunk only creates the folders inside it. Note down which one you
use, since downstream notebooks will use this folder as well.

If a folder already exists, `dir.create` prints a warning. That's harmless.

The configuration file is a different matter: it lives in the cloned repository, which is not
necessarily inside `base_dir`.

In [ ]:
# Directories
# Base directory
base_dir <- "/PATH/TO/YOUR/FOLDER"

# Check that the base directory is defined and exist
stopifnot(dir.exists(base_dir))

# Where you cloned the Metemgee repository. Not necessarily inside base_dir
repo_dir <- "/PATH/TO/YOUR/REPO"

# Data
data_dir <- file.path(base_dir, "data")
dir.create(data_dir)

# Sequences
seq_dir <- file.path(data_dir, "raw_sequences")
dir.create(seq_dir)

# Out
detaxizer_dir <- file.path(data_dir, "detaxizer")
dir.create(detaxizer_dir)

# sheets dir
sheets_dir <- file.path(data_dir, "sheets")
dir.create(sheets_dir)

# Nextflow intermediate files. Can be deleted once the run is done
nextflow_dir <- file.path(data_dir, "nextflow_work")
dir.create(nextflow_dir)

# Software
conda_env <- "Nextflow"
config_file <- file.path(repo_dir, "config/detaxizer.config")

## Test files

For the present example, we'll use publicly available metagenome files. They 
correspond to multiple sequencing runs of two samples; this means that the same
sample was sequenced multiple times to achieve the desired sequencing depth. 
We'll process these files independently; you will find a section on merging
multiple sequencing runs of the same sample at the end of this notebook.

If you are working with your own data, you can skip the next two chunks. Just point `seq_dir` above to
the folder where your `fastq` files are stored.

These are twelve metagenome files, so the download can take a while depending on your connection.

In [ ]:
# URL of the files from the ENA
example_fastqs <- c(
    "ftp://ftp.sra.ebi.ac.uk/vol1/run/ERR108/ERR10880517/MI-142-H.R1.RUN0129.L7.fastq.gz",
    "ftp://ftp.sra.ebi.ac.uk/vol1/run/ERR108/ERR10880518/MI-142-H.R1.RUN0118.L4.fastq.gz",
    "ftp://ftp.sra.ebi.ac.uk/vol1/run/ERR108/ERR10880517/MI-142-H.R2.RUN0129.L7.fastq.gz",
    "ftp://ftp.sra.ebi.ac.uk/vol1/run/ERR108/ERR10880518/MI-142-H.R2.RUN0118.L4.fastq.gz",
    "ftp://ftp.sra.ebi.ac.uk/vol1/run/ERR108/ERR10880579/MI-237-H.R2.RUN0129.L7.fastq.gz",
    "ftp://ftp.sra.ebi.ac.uk/vol1/run/ERR108/ERR10880581/MI-237-H.R1.RUN0118.L2.fastq.gz",
    "ftp://ftp.sra.ebi.ac.uk/vol1/run/ERR108/ERR10880577/MI-237-H.R1.RUN0173.L6.fastq.gz",
    "ftp://ftp.sra.ebi.ac.uk/vol1/run/ERR108/ERR10880582/MI-237-H.R1.RUN0102.L5.fastq.gz",
    "ftp://ftp.sra.ebi.ac.uk/vol1/run/ERR108/ERR10880579/MI-237-H.R1.RUN0129.L7.fastq.gz",
    "ftp://ftp.sra.ebi.ac.uk/vol1/run/ERR108/ERR10880581/MI-237-H.R2.RUN0118.L2.fastq.gz",
    "ftp://ftp.sra.ebi.ac.uk/vol1/run/ERR108/ERR10880577/MI-237-H.R2.RUN0173.L6.fastq.gz",
    "ftp://ftp.sra.ebi.ac.uk/vol1/run/ERR108/ERR10880582/MI-237-H.R2.RUN0102.L5.fastq.gz"
)

example_fastqs

Files that are already there are skipped, so you can run this chunk again without downloading
everything a second time.

In [ ]:
# Download files
# This will take a few minutes
map(example_fastqs, function(url) {
  if (file.exists(file.path(seq_dir, basename(url)))) {
    print("File exists")
  } else {
    print(basename(url))
    download.file(
      url = url,
      destfile = file.path(seq_dir, basename(url)),
      method = "wget",
      extra = "-q"
    )
  }
})

## Create Samples file

We need to tell the pipeline which files correspond to which sample and where in our machine those
files are stored. We do this by creating a table where we specify the sample name and the location of
the forward and reverse `fastq` files.

The pipeline detects on its own whether the reads are single- or paired-end, and whether they are
short or long, from which columns are filled in. Since we have paired short reads, we fill
`short_reads_fastq_1` and `short_reads_fastq_2` and leave `long_reads_fastq_1` empty.

**Note** that you can create this table by hand using Excel or a text editor, and export it as a `csv`
file. In this example we're doing it programmatically, to take the sample name from the full path of
the files.

In [ ]:
# List raw sequences
# This assumes "R1" and "R2" appear in the file names, which is true for the
# example data. Adapt the patterns if your files are named differently.
raw_seq_list <- list.files(seq_dir, pattern = "fastq.gz", full.names = TRUE)

# Forward reads
forward_reads <- raw_seq_list |>
    str_subset("R1")

# Reverse reads
reverse_reads <- raw_seq_list |>
    str_subset("R2")

If your sequencing data is stored in multiple folders, you can concatenate multiple calls to `list.files()`, for example:

```r
# Define dirs
seq_dir_1 = "/PATH/TO/DIR_1"
seq_dir_2 = "/PATH/TO/DIR_2"

# List files
raw_seq_list_1 = list.files(seq_dir_1,  
        pattern = "fastq.gz",
        full.names = TRUE)

raw_seq_list_2 = list.files(seq_dir_2,  
        pattern = "fastq.gz",
        full.names = TRUE)

# Combine
raw_seq_list = c(raw_seq_list_1, raw_seq_list_2)
```

Then, you can continue with separating the forward and reverse files as in the second half of the chunk above

In [ ]:
# Create a single data frame for detaxizer
samples_table <- data.frame(
    short_reads_fastq_1 = forward_reads, # Full path of forward reads
    short_reads_fastq_2 = reverse_reads, # Full path of reverse reads
    long_reads_fastq_1 = ""
) |>
    mutate(
        sample = basename(short_reads_fastq_1),
    ) |>
    relocate(sample) # Reorder columns

# Print head of table
samples_table |>
    head()

Check the `sample` column before continuing. The chunk below saves the table as a comma-separated
file, which is what the pipeline takes as input.

In [ ]:
# Write file
detaxizer_samplesfile <- file.path(sheets_dir, "Example_detaxizer_samples.csv")
samples_table |>
    write_csv(detaxizer_samplesfile)

## Execute pipeline

The code below constructs the bash command to activate the conda environment, change to the working
directory, and run the `detaxizer` pipeline with all required arguments and resources.

What the arguments do:

- `--classification_kraken2` and `--classification_bbduk`: two ways of labelling host reads. Running
  both means the results can be compared and combined
- `--tax2filter`: the taxon to look for and remove. Here we use the genus `Homo` rather than the
  species, so that reads assigned only to the genus are also caught
- `--genome GRCh38`: the reference genome the candidate reads are checked against
- `--preprocessing`: trims adapters and filters low quality reads
- `--reads_minlength 75`: discards reads shorter than this after trimming
- `--fastp_eval_duplication`: reports how many duplicate reads are in the libraries
- `--generate_downstream_samplesheets`: writes ready-made samples files for the pipelines that come
  next, so you don't have to build them by hand
- `-profile` and `-c`: cluster settings and resource allocation
- `-work-dir`: where the intermediate files are stored

If your host is not human, both `--tax2filter` and `--genome` need to be changed.

In [ ]:
# Create command
detaxizer_cmd <- str_glue(
  "conda activate {{conda_env}} && \\
  cd {{out_dir}} && \\
  nextflow run nf-core/detaxizer -r 1.3.0 \\
  -profile m3c \\
  --input {{samples_file}} \\
  --outdir {{out_dir}} \\
  -c {{config_file}} \\
  -work-dir {{nextflow_dir}} \\
  --save_intermediates \\
  --classification_kraken2 \\
  --classification_bbduk \\
  --generate_downstream_samplesheets \\
  --tax2filter Homo \\
  --genome GRCh38 \\
  --preprocessing \\
  --reads_minlength 75 \\
  --fastp_eval_duplication \\
  --fastp_cut_mean_quality 15 \\
  --fastp_qualified_quality 15"
)

Now we can replace the placeholders in the command with the actual paths defined above. The chunk
prints the full command for you to copy and run in your terminal.

In [ ]:
# Fill command
detaxizer_filled <- str_glue(
  detaxizer_cmd,
  out_dir = detaxizer_dir,
  conda_env = conda_env,
  samples_file = detaxizer_samplesfile,
  nextflow_dir = nextflow_dir,
  config_file = config_file
)

# Print command
detaxizer_filled

## While it runs

The pipeline takes several hours and stops if your connection to the cluster drops. Start it inside a
`tmux` or `screen` session so it keeps running when you close the terminal.

Nextflow prints one line per step. If a step fails, adding `-resume` to the command restarts the run
from that point instead of from the beginning.

## What you should have at the end

Inside `data/detaxizer`:

- the filtered `fastq` files, which are the input of every notebook that follows
- the classification and summary tables, with how many reads were labelled as host by each tool
- the samples files written by `--generate_downstream_samplesheets`
- the `multiqc` report, which is the one to open first

For a gut metagenome, the fraction of host reads is usually small. A large fraction is worth looking
into rather than ignoring: it can point to a problem during sample collection or library preparation.

Once you are satisfied with the results, `data/nextflow_work` can be deleted.

## Concatenate runs (optional)

In our example, each sample was sequenced more than once. This can happen when a 
sample is sequenced multiple times to increase the sequencing depth. For downstream
 analyses, we might want to merge runs corresponding to a given sample into a 
 single file.

Other pipelines, such as `taxprofiler` can perform run merging. You can choose to
use that instead. We're doing this here because this allows the concatenated sequences
to be used in other downstream steps that might not have this step built in.

First, we'll load the samples file generated by `detaxizer` and use it to specify
which files should be concatenated.

In [ ]:
# Make sure the output table exists before moving forward
clean_samples_file <- file.path(
  detaxizer_dir,
  "downstream_samplesheets/taxprofiler.csv"
)

stopifnot(file.exists(clean_samples_file))

In [ ]:
# Load clean samples file
clean_samples_table = clean_samples_file |> 
  read_csv()

clean_samples_table |> 
  head()

We can then use this file to determine which files correspond to the same sample
and run a simple script that combines the corresponding forward and reverse files.
We will save the files in a new directory within the `detaxizer` output folder.

In [ ]:
# Create new output directory
concatenated_dir = file.path(detaxizer_dir, "concatenated")
dir.create(concatenated_dir)

# Create empty sequencing depth file
seq_depth_file = file.path(concatenated_dir, "Sequencing_depth.csv")
seq_depth_file |> 
  file.create()

We will use a table with each sample name and the files that need to be merged.
You can create this file manually or programmatically.
You might have to modify the chunk below according to your case.

In [ ]:
# Create a list of files to concatenate by sample name
concatenate_list <- clean_samples_table |>
  mutate(sample = str_remove(sample, "\\.R1.*")) |>
  group_split(sample) |>
  map(function(df) {
    # Extract sample name and create output name
    sample_name <- unique(df$sample)

    # Output files
    combined_file_1 <- file.path(
      concatenated_dir,
      str_c(sample_name, "_merged.R1.fastq.gz")
    )
    combined_file_2 <- file.path(
      concatenated_dir,
      str_c(sample_name, "_merged.R2.fastq.gz")
    )

    if (file.exists(combined_file_1) | file.exists(combined_file_2)) {
      stop("The output files already exist! The notebook will stop")
    } else {
      # lists of files to be concatenated
      fastqs_1 <- str_c(df$fastq_1, collapse = " ")
      fastqs_2 <- str_c(df$fastq_2, collapse = " ")

      # Create data frame
      tibble(
        separated_files = c(fastqs_1, fastqs_2),
        cat_files = c(combined_file_1, combined_file_2)
      ) |>
        mutate(sample = sample_name)
    }
  })

# Create array job data frame
concatenate_table <- concatenate_list |>
  list_rbind() |>
  mutate(ArrayTaskID = row_number()) |>
  relocate(ArrayTaskID, sample)

# # Show top of table
concatenate_table |>
  head()

Inspect the table so that the names of the separated files and concatenated files 
are correct. If the output files already exist, the construction of the table 
will fail. This helps prevent the loss of data by accidentally rewriting the files.


In [ ]:
# Write samples table to file
# Note that this has to be a cvs
concatenate_samplesfile <- file.path(
  sheets_dir,
  "Example_concatenate_samples.csv"
)
write_csv(concatenate_table, file = concatenate_samplesfile)

The chunk below is the template of the script. It might seem long but most of it
is just giving instructions to the cluster. Everything in square brackets is a 
placeholder that we fill in afterwards, so there's nothing to edit here. 

In [ ]:
# Template slurm file
# Do not modify this chunk. The values in square brackets are filled in below.
cat_slurm_raw = str_glue(.open = "[", .close = "]",
"#!/bin/bash
##############################
#       Parameters           #
##############################

# This section tells the cluster what resources your job will need.
# These values are set in the notebook, in the chunk that fills this template.

# Name of the job
#SBATCH --job-name=[[job_name]]

# Generate an output file and give it a name
# This is the log of the run: check it if something goes wrong
#SBATCH --output=%x-%j.out

# Number of tasks
#SBATCH --ntasks=1

# Number of cpus that this task will need
#SBATCH --cpus-per-task=1

# Specify the total memory required per node
#SBATCH --mem=[[memory]]

# Specify the maximum time this job can take to run before being killed (hh:mm:ss)
#SBATCH --time=23:59:00

# Specify number of array jobs
#SBATCH --array=[[array_jobs]]
 
# Specify the path to the config file
samples_file=[[samples_file]]    

# Extract the sample name
sample=$(awk -F, -v ArrayTaskID=$SLURM_ARRAY_TASK_ID '$1==ArrayTaskID {print $2}' $samples_file)

# Extract the files to be concatenated 
separated_files=$(awk -F, -v ArrayTaskID=$SLURM_ARRAY_TASK_ID '$1==ArrayTaskID {print $3}' $samples_file)

# Extract the name of the concatenated file 
cat_file=$(awk -F, -v ArrayTaskID=$SLURM_ARRAY_TASK_ID '$1==ArrayTaskID {print $4}' $samples_file)

# job information
scontrol show job ${SLURM_JOB_ID}
pwd

# per node
# do your real computation

source $HOME/.bashrc

# Set pipe fail
set -o pipefail

cat ${separated_files} > ${cat_file}

# Count number of reads and write to file
seq_depth=$(zcat ${cat_file} | awk 'END {print NR/4}')
echo ${sample},${cat_file},${seq_depth} >> [[seq_depth_file]]
")

Now we can replace the placeholders in the slurm script template with the actual 
paths and filenames defined above. The parameters worth knowing are:
- `memory`: the RAM assigned to each process.
- `seq_depth_file`: file where the count of reads of each file will be stored

In [ ]:
cat_slurm <- str_glue(
    cat_slurm_raw,
    job_name = "fastq_concatenate",
    memory = "8G",
    array_jobs = str_c("1-", nrow(concatenate_table)), # number of array jobs should be expressed as 1-<number of samples to run>, if 10 samples, 1-10
    samples_file = concatenate_samplesfile,
    seq_depth_file = seq_depth_file,
    .open = "[",
    .close = "]"
)

cat_slurm %>%
    print()

The filled template can now be saved to a script to be submitted to the cluster using the `sbatch`
command.

In [ ]:
# Write slurm file
cat_slurmfile <- file.path(sheets_dir, "Concatenate_slurm.sh")
write_lines(cat_slurm, cat_slurmfile)

The following chunk prints the full command for you to copy and run in your terminal.


In [ ]:
# Execution command
str_glue("cd {concatenated_dir} && sbatch {cat_slurmfile}")

Once finished, you should have the the concatenated `fastq.gz` files by sample
in the `data/detaxizer/concatenated` folder. These files can be used as input 
for the notebook that follows.

## Next

Continue with the taxonomic profiling notebook, using the same `base_dir` as here. From this point on,
the notebooks work with the clean reads produced above, not with the raw sequences.